<a href="https://colab.research.google.com/github/paultanisha047-pixel/credit-risk-scoring-engine/blob/main/03_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install boto3 pandas numpy scikit-learn dagshub mlflow pyarrow -q

from google.colab import userdata
import os, boto3, io
import pandas as pd
import numpy as np

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
BUCKET = userdata.get('s3_BUCKET')

s3  = boto3.client('s3')
obj = s3.get_object(Bucket=BUCKET, Key='processed/fct_loans.parquet')
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))

print(f"Loaded {len(df):,} rows ✅")


Loaded 1,345,310 rows ✅


In [3]:
df_train = df[df['is_train'] == True].copy()
df_test  = df[df['is_train'] == False].copy()

print(f"Train: {len(df_train):,} rows | Default rate: {df_train['default_flag'].mean():.1%}")
print(f"Test : {len(df_test):,} rows  | Default rate: {df_test['default_flag'].mean():.1%}")

Train: 826,604 rows | Default rate: 18.4%
Test : 518,706 rows  | Default rate: 22.4%


In [4]:
def compute_iv(df, feature, target, bins=10):
    """
    Compute Information Value for a numeric feature.
    Bins the feature, then computes WoE and IV per bin.
    """
    # Handle nulls — fill with -999 so they form their own bin
    temp = df[[feature, target]].copy()
    temp[feature] = temp[feature].fillna(-999)

    # Create bins — use quantile bins for better distribution
    try:
        temp['bin'] = pd.qcut(temp[feature], q=bins, duplicates='drop')
    except Exception:
        temp['bin'] = pd.cut(temp[feature], bins=bins)

    grouped = temp.groupby('bin')[target].agg(['sum', 'count'])
    grouped.columns = ['defaults', 'total']
    grouped['non_defaults'] = grouped['total'] - grouped['defaults']

    # Total events and non-events
    total_defaults     = grouped['defaults'].sum()
    total_non_defaults = grouped['non_defaults'].sum()

    # WoE = ln(dist_non_defaults / dist_defaults)
    grouped['dist_defaults']     = grouped['defaults'] / total_defaults
    grouped['dist_non_defaults'] = grouped['non_defaults'] / total_non_defaults

    # Add small epsilon to avoid log(0)
    eps = 1e-10
    grouped['woe'] = np.log(
        (grouped['dist_non_defaults'] + eps) / (grouped['dist_defaults'] + eps)
    )

    # IV = sum((dist_non_defaults - dist_defaults) * WoE)
    grouped['iv'] = (grouped['dist_non_defaults'] - grouped['dist_defaults']) * grouped['woe']

    return grouped['iv'].sum()

# Compute IV for all numeric features
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols
                if c not in ['default_flag', 'is_train', 'loan_id']]

print("Computing Information Value for all features...")
iv_scores = {}
for col in numeric_cols:
    try:
        iv_scores[col] = compute_iv(df_train, col, 'default_flag')
    except Exception as e:
        iv_scores[col] = 0.0

iv_df = pd.DataFrame.from_dict(iv_scores, orient='index', columns=['IV'])\
          .sort_values('IV', ascending=False)

print("\nFeature IV scores:")
print(iv_df.to_string())

Computing Information Value for all features...


/tmp/ipykernel_679/2460648923.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp.groupby('bin')[target].agg(['sum', 'count'])
/tmp/ipykernel_679/2460648923.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp.groupby('bin')[target].agg(['sum', 'count'])
/tmp/ipykernel_679/2460648923.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp.groupby('bin')[target].agg(['sum', 'coun


Feature IV scores:
                                   IV
sub_grade_encoded            0.492935
int_rate                     0.454480
grade_encoded                0.414749
fico_score                   0.119297
acc_open_past_24mths         0.082140
dti                          0.074874
bc_open_to_buy               0.046092
avg_cur_bal                  0.044586
tot_hi_cred_lim              0.038277
funded_amnt                  0.036903
loan_amnt                    0.036845
tot_cur_bal                  0.035728
total_bc_limit               0.034250
percent_bc_gt_75             0.032474
log_annual_inc               0.031307
bc_util                      0.028616
installment                  0.026017
total_rev_hi_lim             0.024572
revol_util                   0.022093
mort_acc                     0.019822
inq_last_6mths               0.018250
credit_age_years             0.014555
num_op_rev_tl                0.013382
num_sats                     0.011948
num_actv_bc_tl               0

In [5]:
selected_features = iv_df[iv_df['IV'] >= 0.02].index.tolist()

# Categorical features to encode separately in training pipeline
categorical_features = ['home_ownership', 'verification_status', 'purpose']

# Add categoricals back if not already in selected list
for cat in categorical_features:
    if cat not in selected_features:
        selected_features.append(cat)

print(f"Selected {len(selected_features)} features (IV >= 0.02):")
for f in selected_features:
    iv = iv_scores.get(f, 'categorical')
    print(f"  {f:<35} IV={iv:.4f}" if isinstance(iv, float) else f"  {f:<35} categorical")

# Save selected features list to S3 for use in training notebook
import json
feature_config = {
    'selected_features': selected_features,
    'categorical_features': categorical_features,
    'numeric_features': [f for f in selected_features
                         if f not in categorical_features]
}

s3.put_object(
    Bucket=BUCKET,
    Key='processed/feature_config.json',
    Body=json.dumps(feature_config, indent=2)
)
print("\nfeature_config.json saved to S3 ✅")


Selected 22 features (IV >= 0.02):
  sub_grade_encoded                   IV=0.4929
  int_rate                            IV=0.4545
  grade_encoded                       IV=0.4147
  fico_score                          IV=0.1193
  acc_open_past_24mths                IV=0.0821
  dti                                 IV=0.0749
  bc_open_to_buy                      IV=0.0461
  avg_cur_bal                         IV=0.0446
  tot_hi_cred_lim                     IV=0.0383
  funded_amnt                         IV=0.0369
  loan_amnt                           IV=0.0368
  tot_cur_bal                         IV=0.0357
  total_bc_limit                      IV=0.0343
  percent_bc_gt_75                    IV=0.0325
  log_annual_inc                      IV=0.0313
  bc_util                             IV=0.0286
  installment                         IV=0.0260
  total_rev_hi_lim                    IV=0.0246
  revol_util                          IV=0.0221
  home_ownership                      categorical
  v